In [ ]:
# library imports
import pandas as pd                             # tables, dataframes
import numpy as np                              # arrays, numerical computations
import matplotlib.pyplot as plt                 # plots
from uncertainties import ufloat as uf          # uncertainty propagation
from scipy.optimize import curve_fit            # fitting

from pathlib import Path                        # paths in operating system
# path config
downloads_dir = Path.home() / 'Downloads'
# make sure the folder exists (just in case)
downloads_dir.mkdir(parents=True, exist_ok=True)
# output_path = downloads_dir / 'my_plot.pdf'

In [ ]:
# custom functions

# type A uncertainty
def unc_A(data):
    '''
    Type A uncertainty (standard error of the mean)

    data : list or numpy array
        measurements
    return : float
        type A uncertainty
    '''
    data = np.array(data)
    n = len(data)

    s = np.std(data, ddof=1)
    u_A = s / np.sqrt(n)

    return u_A

# loading from df into individual arrays
def df_to_arrays(df, column, first_row=0, last_row=None, dtype=float):
    '''
    Extracts the selected columns of a dataframe as separate numpy arrays.
    NaN values are dropped from each column.

    df : dataframe;
        source dataframe
    column : str or list[str];
        column name (str) or list of column names (list[str])
    first_row : int;
        first row (index) to include
    last_row : int or None;
        row (index) at which to stop (exclusive); None = until the end
    dtype : type;
        data type of the resulting arrays (default float)
    return : list[numpy.ndarray];
        one array per column
    '''
    if isinstance(column, str):
        column = [column]

    missing = [s for s in column if s not in df.columns]
    if missing:
        raise KeyError(f'Missing columns: {missing}')

    return [df[s].iloc[first_row:last_row].dropna().to_numpy(dtype=dtype) for s in column]

# ufloat from the arithmetic mean
def ufloat_unc_A(data):
    '''
    Converts a dataset to a ufloat (arithmetic mean +- type A uncertainty)

    data : list or numpy array
        dataset of measurements
    return : ufloat
    '''

    return uf(np.mean(data), unc_A(data))

# conversion from popt and pcov to ufloats
from uncertainties import correlated_values     # popt, pcov, correlations
def popt_pcov_to_ufloat(popt, pcov, correlated=False):
    '''
    Converts the result of curve_fit (popt, pcov) to ufloats.

    popt : array_like
        optimal parameters from curve_fit
    pcov : array_like
        covariance matrix from curve_fit
    correlated : bool
        True  -> ufloats carry correlations between parameters
        False -> independent ufloats (only sqrt of the pcov diagonal)
    return : tuple(ufloat, ...)
        parameters with uncertainties
    '''
    popt = np.asarray(popt, dtype=float)
    pcov = np.asarray(pcov, dtype=float)

    if correlated:
        return tuple(correlated_values(popt, pcov))

    perr = np.sqrt(np.diag(pcov))

    return tuple(uf(v, e) for v, e in zip(popt, perr))

# uarray to LaTeX
def uarray_to_latex(*array, digit=1, format='u', style='L', only_return=False):
    '''
    Formats ufloats as LaTeX strings (e.g. $1.23 \\pm 0.04$)

    *array : ufloat
        one or more ufloats to format
    digit : int
        number of digits / precision in the format specification
    format : str
        ufloat format type (e.g. 'u' for the uncertainty in parentheses)
    style : str
        output style; 'L' for LaTeX
    only_return : bool
        True  -> return the list of strings
        False -> print each string
    return : list[str] or None
        formatted strings if only_return=True, otherwise None
    '''
    output = [f'${u:.{digit}{format}{style}}$' for u in array]
    if only_return:
        return output
    else:
        for s in output:
            print(s)

In [ ]:
# globalni funkce a parametry

UG0 = 3.7 # Volt
UD0 = 11.0 # Volt

# Barkhausenova rovnice
def Barkhausen(S, Ri):
    mu = S*Ri
    return mu

# linearni model
def lin_model(x, A, B):
    return A*x + B

In [ ]:
# loading Google Sheets
sheet_id = "1yAYdf4AN5eZNSYrIgae-Ci-KBFvSRCxHeFl2A0DHcYU"
gid = "0"
url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv&gid={gid}"
df_UG_characteristic = pd.read_csv(url)

In [ ]:
# prevodni charakteristika

# nacteni hodnot
UG_UD0, ID_UD0_mm = df_to_arrays(df_UG_characteristic, ['Napětí hradla UG (V)', 'Proud drain ID (mA)'])
ID_UD0 = ID_UD0_mm*1e-3
UG0_mask = (UG_UD0 <= (UG0 + 0.1)) & (UG_UD0 >= (UG0 - 0.1))

# fit
popt, pcov = curve_fit(lin_model, UG_UD0[UG0_mask], ID_UD0[UG0_mask])
A, _ = popt_pcov_to_ufloat(popt, pcov)
S = A
print('Strmost S pro pracovní bod je:', S.n)
uarray_to_latex(S)

# vykresleni grafu
fig, ax = plt.subplots(figsize=(16, 9))
ax.set_ylabel(r'$I_D$ [mA]', fontsize=20)
ax.set_xlabel(r'$U_G$ [V]', fontsize=20)
ax.scatter(UG_UD0, ID_UD0*1e3, label='Naměřené hodnoty')
#ax.plot(UG_UD0[UG0_mask], lin_model(UG_UD0[UG0_mask], *popt)*1e3, label='Lineární fit')
ax.scatter(UG0, 72.552, color='r', label='Pracovní bod')
ax.tick_params(labelsize=15)
ax.grid(True, alpha=0.7)
ax.legend(fontsize=15)
plt.savefig(downloads_dir / 'prevodni_char.pdf', dpi=300, bbox_inches='tight')

In [ ]:
# loading Google Sheets
sheet_id = "1yAYdf4AN5eZNSYrIgae-Ci-KBFvSRCxHeFl2A0DHcYU"
gid = "846904353"
url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv&gid={gid}"
df_output_characteristic = pd.read_csv(url)

In [ ]:
# vystupni charaktristika

# nacteni hodnot
UD_UG0, ID_UG0_mm = df_to_arrays(df_output_characteristic, ['Napětí drain UD (V)', 'Proud drain ID (mA)'])
ID_UG0 = ID_UG0_mm*1e-3
UD0_mask = (UD_UG0 <= (UD0 + 4)) & (UD_UG0 >= (UD0 - 4))

# fit
popt, pcov = curve_fit(lin_model, ID_UG0[UD0_mask], UD_UG0[UD0_mask])
A, _ = popt_pcov_to_ufloat(popt, pcov)
Ri = A
print('Vnitřní odpor tranzistoru pro pracovní bod je:', Ri.n)
uarray_to_latex(Ri)

# vykresleni grafu
fig, ax = plt.subplots(figsize=(16, 9))
ax.set_ylabel(r'$I_D$ [mA]', fontsize=20)
ax.set_xlabel(r'$U_D$ [V]', fontsize=20)
ax.scatter(UD_UG0, ID_UG0*1e3, label='Naměřené hodnoty')
#ax.plot(lin_model(ID_UG0[UD0_mask], *popt), ID_UG0[UD0_mask]*1e3, label='Lineární fit')
ax.scatter(UD0, 74.077, color='r', label='Pracovní bod')
ax.tick_params(labelsize=15)
ax.grid(True, alpha=0.7)
ax.legend(fontsize=15)
plt.savefig(downloads_dir / 'vystupni_char.pdf', dpi=300, bbox_inches='tight')

In [ ]:
# loading Google Sheets
sheet_id = "1yAYdf4AN5eZNSYrIgae-Ci-KBFvSRCxHeFl2A0DHcYU"
gid = "604032957"
url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv&gid={gid}"
df_UG_characteristic_PC = pd.read_csv(url)

In [ ]:
# prevodni charakteristika PC

# nacteni dat
UG_UD0_PC, ID_UD0_PC_mm = df_to_arrays(df_UG_characteristic_PC, ['napeti_Ug_(V)', 'proud_Id_(mA)'])
ID_UD0_PC = ID_UD0_PC_mm*1e-3
UG0_mask_PC = (UG_UD0_PC <= (UG0 + 0.04)) & (UG_UD0_PC >= (UG0 - 0.041))

# fit
popt, pcov = curve_fit(lin_model, UG_UD0_PC[UG0_mask_PC], ID_UD0_PC[UG0_mask_PC])
A, _ = popt_pcov_to_ufloat(popt, pcov)
S = A
print('Strmost S pro pracovní bod je:', S.n)
uarray_to_latex(S)

# vykresleni grafu
fig, ax = plt.subplots(figsize=(16, 9))
ax.set_ylabel(r'$I_D$ [mA]', fontsize=20)
ax.set_xlabel(r'$U_G$ [V]', fontsize=20)
ax.plot(UG_UD0_PC, ID_UD0_PC*1e3, label='Naměřené hodnoty')
ax.scatter(3.699999999999965, 74.4021899, color='r', label='Pracovní bod')
ax.tick_params(labelsize=15)
ax.grid(True, alpha=0.7)
ax.legend(fontsize=15)
plt.savefig(downloads_dir / 'prevodni_char_PC.pdf', dpi=300, bbox_inches='tight')

# vykresleni grafu
fig, ax = plt.subplots(figsize=(16, 9))
ax.set_ylabel(r'$I_D$ [mA]', fontsize=20)
ax.set_xlabel(r'$U_G$ [V]', fontsize=20)
ax.set_xlim(3.60, 3.80)
ax.set_ylim(25, 125)
ax.scatter(UG_UD0_PC, ID_UD0_PC*1e3, label='Naměřené hodnoty')
ax.plot(UG_UD0_PC[UG0_mask_PC], lin_model(UG_UD0_PC[UG0_mask_PC], *popt)*1e3, label='Lineární fit')
ax.scatter(3.699999999999965, 74.4021899, color='r', label='Pracovní bod')
ax.tick_params(labelsize=15)
ax.grid(True, alpha=0.7)
ax.legend(fontsize=15)
plt.savefig(downloads_dir / 'prevodni_char_PC_vyrez.pdf', dpi=300, bbox_inches='tight')

In [ ]:
# loading Google Sheets
sheet_id = "1yAYdf4AN5eZNSYrIgae-Ci-KBFvSRCxHeFl2A0DHcYU"
gid = "279767635"
url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv&gid={gid}"
df_output_characteristic_PC = pd.read_csv(url)
df_output_characteristic_PC

In [ ]:
# vystupni charaktristika PC

# nacteni hodnot
output_char_PC_list = ['UD36', 'ID36', 'UD365', 'ID365', 'UD37', 'ID37', 'UD375', 'ID375', 'UD38', 'ID38']
UD_UG36_PC, ID_UG36_PC, UD_UG365_PC, ID_UG365_PC, UD_UG37_PC, ID_UG37_PC_mm, UD_UG375_PC, ID_UG375_PC, UD_UG38_PC, ID_UG38_PC = df_to_arrays(df_output_characteristic_PC, output_char_PC_list)
ID_UG37_PC = ID_UG37_PC_mm*1e-3
UG0_mask_PC = (UD_UG37_PC <= (UD0 + 4)) & (UD_UG37_PC >= (UD0 - 4))

# fit
popt, pcov = curve_fit(lin_model, ID_UG37_PC[UG0_mask_PC], UD_UG37_PC[UG0_mask_PC])
A, _ = popt_pcov_to_ufloat(popt, pcov)
Ri = A
print('Vnitřní odpor tranzistoru pro pracovní bod je:', Ri.n)
uarray_to_latex(Ri)

# vykresleni grafu
fig, ax = plt.subplots(figsize=(16, 9))
ax.set_ylabel(r'$I_D$ [mA]', fontsize=20)
ax.set_xlabel(r'$U_D$ [V]', fontsize=20)
UD_dic = {'3.60': UD_UG36_PC, '3.65': UD_UG365_PC, '3.70': UD_UG37_PC, '3.75': UD_UG375_PC, '3.80': UD_UG38_PC}
ID_dic = {'3.60': ID_UG36_PC, '3.65': ID_UG365_PC, '3.70': ID_UG37_PC*1e3, '3.75': ID_UG375_PC, '3.80': ID_UG38_PC}
for i in reversed(list(UD_dic)):
    ax.plot(UD_dic[i], ID_dic[i], label=fr'Naměřené hodnoty: $U_{{G0}} = {i}\,V$')
ax.scatter(11.00000000000002, 79.4428168, color='r', label='Pracovní bod')
ax.plot(lin_model(ID_UG37_PC[UG0_mask_PC], *popt), ID_UG37_PC[UG0_mask_PC]*1e3, c='red', label='Lineární fit')
ax.tick_params(labelsize=15)
ax.grid(True, alpha=0.7)
ax.legend(fontsize=15)
plt.savefig(downloads_dir / 'vystupni_char_PC.pdf', dpi=300, bbox_inches='tight')


# vykresleni grafu
fig, ax = plt.subplots(figsize=(16, 9))
ax.set_ylabel(r'$I_D$ [mA]', fontsize=20)
ax.set_xlabel(r'$U_D$ [V]', fontsize=20)
ax.set_xlim(5, 18)
ax.set_ylim(40, 150)
ax.scatter(UD_dic['3.75'], ID_dic['3.75'], s=7, c='tab:orange', label=fr'Naměřené hodnoty: $U_{{G0}} = 3.75\,$V')
ax.scatter(UD_dic['3.70'], ID_dic['3.70'], s=7, c='tab:green', label=fr'Naměřené hodnoty: $U_{{G0}} = 3.70\,$V')
ax.scatter(UD_dic['3.65'], ID_dic['3.65'], s=7, c='tab:red', label=fr'Naměřené hodnoty: $U_{{G0}} = 3.65\,$V')
ax.scatter(11.00000000000002, 79.4428168, color='r', label='Pracovní bod')
ax.plot(lin_model(ID_UG37_PC[UG0_mask_PC], *popt), ID_UG37_PC[UG0_mask_PC]*1e3, c='red', linewidth=1.5, label='Lineární fit')
UD_linspace = np.linspace(0, 25, 1000)
E_amplt = 22
Rz = 150
Rz_teor = Rz
Rz_teor = 11_000.00000000002/79.4428168
zatezovaci_primka = (E_amplt - UD_UG365_PC)/Rz_teor
rozdil_365 = ID_UG365_PC - zatezovaci_primka*1e3
i_arg_365 = np.argmin(np.abs(rozdil_365))
print(f"Nejbližší bod: x = {UD_UG365_PC[i_arg_365]}, y = {ID_UG365_PC[i_arg_365]}, odchylka = {rozdil_365[i_arg_365]:.3f}")
ax.scatter(UD_UG365_PC[i_arg_365], ID_UG365_PC[i_arg_365], s=100, c='blue', marker='x')

zatezovaci_primka = (E_amplt - UD_UG375_PC)/Rz_teor
rozdil_375 = ID_UG375_PC - zatezovaci_primka*1e3
i_arg_375 = np.argmin(np.abs(rozdil_375))
print(f"Nejbližší bod: x = {UD_UG375_PC[i_arg_375]}, y = {ID_UG375_PC[i_arg_375]}, odchylka = {rozdil_375[i_arg_375]:.3f}")
ax.scatter(UD_UG375_PC[i_arg_375], ID_UG375_PC[i_arg_375], s=100, c='blue', marker='x')

delta_ug = UD_UG365_PC[i_arg_365] - UD_UG375_PC[i_arg_375]
print('Delta UG je:', delta_ug)
A_G = delta_ug/0.1
print('Graficky určené zesílení je:', A_G)

ax.plot(UD_UG365_PC, zatezovaci_primka*1e3, label='Zatěžovací přímka:\n'fr'$E = {E_amplt}\,$V, $R_z = {Rz_teor:.0f}\,\Omega$')

ax.tick_params(labelsize=15)
ax.grid(True, alpha=0.7)
ax.legend(fontsize=15)
plt.savefig(downloads_dir / 'vystupni_char_PC_vyrez.pdf', dpi=300, bbox_inches='tight')


mu = Barkhausen(S, Ri)
print('Zesilovací činitel tranzistoru je:', mu.n)
uarray_to_latex(mu)

In [ ]:
# zesileni

E_amplt = 22.0
Rz = 150
Rz = Rz_teor
U1 = 0.06231
U2 = -6.023
A_M = U2/U1
print('Naměřené zesílení je:', A_M)

A_V = (-mu)/(1+(Ri/Rz))
print('Vypočítané zesílení je:', A_V.n)
uarray_to_latex(A_V)

print('Graficky určené zesílení je:', -A_G)